# 02 - Chunking: Three strategies + a general retrieval coverage tool

**Aligns with**: S4 Sec. 4 | **Estimated time**: 45 minutes | **Estimated cost**: ~$0.05

## The question 00_quickstart left unanswered: is chunk_size=500 optimal? How do you know?

Two things in this notebook:

1. **Build a general tool**, `CoverageDiagnostic` - quantify "did your retrieval
   actually pull the relevant content?" for any corpus, any chunker
2. **Use the tool** to compare 3 chunking strategies on WF / Tesla / AMD

> **Design choice**: we did NOT hard-code the fact "WF lost 8 tables" as a demo.
> We wrote a diagnostic function that runs on **any corpus**, then **used WF as
> the example**. So when a student switches documents (e.g., a Microsoft 10-K
> for an interview demo), the whole notebook still applies.


In [4]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
from dotenv import load_dotenv; load_dotenv()

from src.pipelines.ingestion import IngestionPipeline
from src.chunkers import FixedSizeChunker, RecursiveChunker, ParentChildChunker
from src.retrievers import VectorRetriever
from src.evaluators.coverage import CoverageDiagnostic, compare_strategies
import pandas as pd

## Step 1: Ingest Wells Fargo (cache hit, should return instantly)

In [5]:
pipeline = IngestionPipeline()
report = pipeline.ingest(str(ROOT / "data/uploads/wells_fargo.pdf"))
doc = report.document
print(f"Title: {doc.title}")
print(f"  {len(doc.blocks)} blocks total")
print(f"  {report.n_table_blocks} table blocks (note: PyMuPDF may report 0 because WF tables are positioned text, not native table structures)")
print(f"  parse_cache_hit = {report.parse_cache_hit}")

Title: Wells Fargo Reports Fourth Quarter 2025 Net Income of $5.4 billion, or $1.62 per Diluted Share
  419 blocks total
  0 table blocks (note: PyMuPDF may report 0 because WF tables are positioned text, not native table structures)
  parse_cache_hit = True


## Step 2: Three chunkers, all class-based

In [6]:
chunkers = {
    "fixed_size_500":  FixedSizeChunker(size=500, overlap=80),
    "recursive_400":   RecursiveChunker(chunk_size=400, overlap=60),
    "parent_child":    ParentChildChunker(parent_size=800, child_size=150),
}

for name, ch in chunkers.items():
    chunks = ch.chunk(doc)
    avg = sum(len(c.text) for c in chunks)//max(1,len(chunks))
    print(f"  {name:18s}: {len(chunks):4d} chunks (avg {avg} chars)")

  fixed_size_500    :   18 chunks (avg 2052 chars)
  recursive_400     :  387 chunks (avg 106 chars)
  parent_child      :   88 chunks (avg 494 chars)


## Step 3: CoverageDiagnostic - what fixed_size actually does on WF

`embeddings_cache_dir` makes the second run free:


In [7]:
test_queries = [
    "What was Wells Fargo's Q4 2025 net income?",
    "What was the diluted EPS in Q4 2025?",
    "What was the CET1 ratio?",
    "What was the average loan balance?",
    "How did Consumer Banking perform?",
    "What were the main drivers of net interest income?",
    "What capital return actions were taken?",
    "How did the segment compositions change?",
]

chunks_fixed = chunkers["fixed_size_500"].chunk(doc)
retriever = VectorRetriever(
    persist_dir=str(ROOT / "tmp_chroma_fixed"),
    collection="lab_02_fixed",
    embeddings_cache_dir=ROOT / "cache/embeddings",
)
retriever.reset()
retriever.index(chunks_fixed)

diag = CoverageDiagnostic(retriever, k=5).diagnose(test_queries)
df = CoverageDiagnostic.to_dataframe(diag)
df

,query,n_retrieved,pct_dense,avg_density,n_unique_pages,avg_chars,top_snippet
0,What was Wells Fargo's Q4 2025 net income?,5,0.4,0.211,3,1866.0,"News Release | January 14, 2026 Wells Fargo Re..."
1,What was the diluted EPS in Q4 2025?,5,0.4,0.183,4,2012.0,"2025 Dec 31, 2024 Earnings (in millions) N..."
2,What was the CET1 ratio?,5,0.4,0.220,4,1888.0,. 4. Return on equity (ROE) represents Wells ...
3,What was the average loan balance?,5,0.4,0.209,5,1891.0,181.1 164.7 160.7 139.1 135.6 11.0 11.1 2...
4,How did Consumer Banking perform?,5,0.6,0.207,4,1858.0,presented on page 9. 3 Operating Segment Per...
5,What were the main drivers of net interest inc...,5,0.6,0.252,5,1739.0,"loan balances, including the impact of the tr..."
6,What capital return actions were taken?,5,0.6,0.205,5,1925.0,on a taxable-equivalent basis 2.60 2.61 2.70 ...
7,How did the segment compositions change?,5,0.8,0.230,4,1812.0,presented on page 9. 3 Operating Segment Per...


**Two columns to focus on**:

- `pct_dense`: of the 5 retrieved chunks, **how many are "data-dense"** (>=20% of
  tokens are numeric - typical of table fragments)
- `avg_density`: average **numeric density** across retrieved chunks (0 = pure
  prose, 1 = all numbers)

**How to read this**: for a fact-style query like "Q4 2025 net income?", you want
**high `pct_dense` (>0.4) and high `avg_density` (>0.15)** - that means the
retriever found the table region. If both metrics are low (<=0.1), retrieval
returned a pile of prose paragraphs and **the chunker shredded the table**.

> **Why not just check "does it contain a number"?** Because every paragraph in
> a financial doc has dates and dollar figures - "contains a number" is almost
> always True. **Density** is the actual signal for whether the chunker preserved
> the table structure. The metric evolution itself is a real engineering lesson
> (start simple, observe saturation, refine).

On WF, fixed_size typically has notably lower `avg_density` than recursive /
parent_child on numeric queries - the positioned-text tables get shredded
across multiple chunks, losing their density.


## Step 4: `compare_strategies` - direct horizontal comparison of all 3 chunkers

In [8]:
def make_retriever():
    """Factory: returns a fresh VectorRetriever with embedding cache."""
    import uuid
    return VectorRetriever(
        persist_dir=str(ROOT / f"tmp_chroma_compare_{uuid.uuid4().hex[:6]}"),
        collection=f"lab_02_compare_{uuid.uuid4().hex[:6]}",
        embeddings_cache_dir=ROOT / "cache/embeddings",
    )

result_df = compare_strategies(
    chunkers=chunkers,
    doc=doc,
    queries=test_queries,
    retriever_factory=make_retriever,
    k=5,
)

pivot = result_df.pivot_table(
    values="avg_density", index="query", columns="strategy", aggfunc="first",
).round(2)
pivot

strategy,fixed_size_500,parent_child,recursive_400
query,,,
How did Consumer Banking perform?,0.21,0.14,0.12
How did the segment compositions change?,0.23,0.15,0.26
What capital return actions were taken?,0.20,0.15,0.06
What was Wells Fargo's Q4 2025 net income?,0.21,0.13,0.13
What was the CET1 ratio?,0.22,0.13,0.20
What was the average loan balance?,0.21,0.16,0.16
What was the diluted EPS in Q4 2025?,0.18,0.19,0.20
What were the main drivers of net interest income?,0.25,0.31,0.07


## Step 5: Same tool applied to Tesla / AMD

In [7]:
tesla_doc = pipeline.ingest(str(ROOT / "data/uploads/tesla.pdf")).document

tesla_queries = [
    "What was Tesla's Q1 2026 vehicle production?",
    "What was the energy storage deployment?",
    "How did automotive margins evolve?",
    "What is the AI/Cortex initiative?",
]

tesla_compare = compare_strategies(
    chunkers=chunkers, doc=tesla_doc, queries=tesla_queries,
    retriever_factory=make_retriever, k=5,
)
tesla_compare.pivot_table(
    values="avg_density", index="query", columns="strategy", aggfunc="first"
).round(2)

strategy,fixed_size_500,parent_child,recursive_400
query,,,
How did automotive margins evolve?,0.26,0.16,0.30
What is the AI/Cortex initiative?,0.04,0.05,0.13
What was Tesla's Q1 2026 vehicle production?,0.21,0.11,0.08
What was the energy storage deployment?,0.09,0.07,0.22


## Step 6: Decision framework

**There is no "best chunking strategy" - only "the best strategy for your query
distribution":**

| Query type | Recommended | Why |
|---|---|---|
| Short numeric lookup | parent_child | Small child precisely hits the block with the number |
| Long narrative | recursive | Preserves heading + section boundaries |
| Generic baseline | recursive | Simple and reliable |
| **Avoid** | fixed_size | Cuts across blocks, drops context |

**Interview talking point**:

> "I didn't hard-code a chunking strategy. I built `compare_strategies` plus the
> `CoverageDiagnostic` quantification function so the team can decide which
> chunker fits a new corpus in 5 minutes. This is measurement-driven engineering."

## Exercise 2.A

Add the AMD PDF (image-heavy) to `compare_strategies`. Predict and verify:
- AMD's PowerPoint style means repeated images, so embedding cache hit rate is high
- AMD data is scattered through charts, so fixed_size does worse on numeric queries


In [ ]:
# TODO: amd_doc = pipeline.ingest(...)
# TODO: amd_compare = compare_strategies(chunkers, amd_doc, [...], make_retriever)
